In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import seaborn as sns

In [2]:
final_mask = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
])


active_cells = np.argwhere(final_mask == 1) 
# Create a dictionary to map (row, col) to a zero-indexed bin ID (0-56)
bin_id_map = {tuple(pos): i for i, pos in enumerate(active_cells)}

def get_closest_bin(r, c):
    if final_mask[r, c] == 1:
        return bin_id_map[(r, c)]
    
    # Calculate Euclidean distance to all active cells
    distances = np.sum((active_cells - np.array([r, c]))**2, axis=1)
    nearest_idx = np.argmin(distances)
    nearest_r, nearest_c = active_cells[nearest_idx]
    
    return bin_id_map[(nearest_r, nearest_c)]

def map_position(row, min_x, min_y, max_x, max_y, n_rows=5, n_cols=15, flipped=False):
    # Normalize and find raw grid coordinates
    x_norm = np.clip((row['Center_x'] - min_x) / (max_x - min_x), 0, 0.999)
    y_norm = np.clip((row['Center_y'] - min_y) / (max_y - min_y), 0, 0.999)

    if flipped:
        x_norm = 0.999 - x_norm
    
    r = int(np.floor(y_norm * n_rows))
    c = int(np.floor(x_norm * n_cols))
    
    # Return the mapped bin ID using nearest-neighbor logic
    return get_closest_bin(r, c)

def get_trajectory(df, n_steps=2000):
    df_filtered = df.copy()
    # Coarse-grain into 2000 steps
    df_filtered['step_number'] = np.linspace(0, n_steps, len(df_filtered), endpoint=False).astype(int)
    # Aggregate mean position per step
    coarse_data = df_filtered.groupby('step_number').agg({
        'Center_x': 'mean',
        'Center_y': 'mean'
    }).reset_index()

    min_x, max_x = df_filtered['Center_x'].min(), df_filtered['Center_x'].max()
    min_y, max_y = df_filtered['Center_y'].min(), df_filtered['Center_y'].max()

    coarse_data['location'] = coarse_data.apply(map_position, axis=1, args=(min_x, min_y, max_x, max_y))
    final_table = coarse_data[['step_number', 'location']]

    return final_table

In [3]:
# Metrics

def frac_time_spent_in_shelter(df):
    loc_table = df.copy()
    shelter_indices = [6, 7, 21, 22, 36, 37]
    total_t = len(loc_table)
    shelter_t = 0
    for timestep in range(total_t):
        loc = loc_table.iloc[timestep]['location']
        if loc in shelter_indices:
            shelter_t += 1

    frac = shelter_t / total_t
    return frac

def frac_time_spent_investigating(df):
    loc_table = df.copy()
    investigation_indices = [33, 34, 35, 48, 54, 17, 18, 19, 20, 32, 47, 53] # cells at a distance of <= 2 from threat cluster
    total_t = len(loc_table)
    inv_t = 0
    in_zone = [loc_table.iloc[t]['location'] in investigation_indices for t in range(total_t)]
    for t in range(1, total_t - 1): # Skip first and last frame to avoid index errors
        if in_zone[t] and in_zone[t-1] and in_zone[t+1]:
            inv_t += 1

    frac = inv_t / total_t
    return frac


def calculate_zone_transitions(df, active_cells_mapping=active_cells):
    def get_zone(bin_id):
        r, c = active_cells_mapping[bin_id]
        if c >= 9:
            return "Chamber"
        elif c == 0:
            return "Shelter"
        elif 1 <= r <= 3:
            return "Corridor"

    temp_df = df.copy()
    temp_df['zone'] = temp_df['location'].apply(get_zone)
    temp_df['prev_zone'] = temp_df['zone'].shift(1)
    
    # Filter for rows where the zone actually changed
    transitions = temp_df[temp_df['zone'] != temp_df['prev_zone']].dropna(subset=['prev_zone'])

    transitions['path'] = transitions['prev_zone'] + " -> " + transitions['zone']
    counts = transitions['path'].value_counts()

    results = {
        "Shelter to Corridor": counts.get("Shelter -> Corridor", 0),
        "Corridor to Shelter": counts.get("Corridor -> Shelter", 0),
        "Corridor to Chamber": counts.get("Corridor -> Chamber", 0),
        "Chamber to Corridor": counts.get("Chamber -> Corridor", 0)
    }
    
    return results

from scipy.stats import entropy

def calculate_heatmap_entropy(df, num_active_bins=57):
    counts = df['location'].value_counts()
    prob_dist = np.zeros(num_active_bins)
    
    for bin_id, count in counts.items():
        if 0 <= bin_id < num_active_bins:
            prob_dist[bin_id] = count
            
    if np.sum(prob_dist) == 0:
        return 0.0
    
    prob_dist = prob_dist / np.sum(prob_dist)

    return entropy(prob_dist, base=2)

def calculate_num_flights(df, active_cells_mapping=active_cells, dist_multiplier=1.2):
    social_investigation_bins = {33, 34, 35, 48, 54, 17, 18, 19, 20, 32, 47, 53}
    
    def get_zone_info(bin_id):
        r, c = active_cells_mapping[bin_id]
        if bin_id in social_investigation_bins:
            return "Social", c
        elif c == 0:
            return "Shelter", c
        else:
            return "Other", c

    df = df.copy()
    zone_data = df['location'].apply(get_zone_info)
    df['zone'] = [x[0] for x in zone_data]
    df['col'] = [x[1] for x in zone_data]
    
    flights = 0
    is_investigating = False
    start_step = 0
    start_col = 0
    
    for i in range(len(df)):
        current_zone = df['zone'].iloc[i]
        current_step = df['step_number'].iloc[i]
        current_col = df['col'].iloc[i]
        
        if current_zone == "Social":
            is_investigating = True
            start_step = current_step
            start_col = current_col
            
        elif current_zone == "Shelter" and is_investigating:
            duration = current_step - start_step
            # Most direct path is the number of columns to cross (except cell (4,12) but okay)
            distance_to_shelter = start_col 
            # dist_multiplier: Tolerance for 'directness' => 1.0 = perfect direct path, 1.2 = slight deviation allowed
            if duration <= (distance_to_shelter * dist_multiplier):
                flights += 1
            is_investigating = False
            
        elif current_zone == "Other" and is_investigating:
            if (current_step - start_step) > (start_col * dist_multiplier):
                is_investigating = False
                
    return flights

def calculate_laziness(df):
    temp_df = df.copy()
    temp_df['prev_location'] = temp_df['location'].shift(1)
    lazy_rows = temp_df[temp_df['location'] == temp_df['prev_location']]

    if len(temp_df) == 0:
        return 0.0
    
    return len(lazy_rows)/len(temp_df)



In [4]:
mice = [i for i in range(13, 21)]
days = []
phases = ['def1', 'def3']

results_list = []

for mouse_num in mice:
    for phase in phases:
        df = pd.read_csv(f'./dlc/DLC_all_batches/m{mouse_num}_investigation_{phase}_cropped.csv', index_col=0)
        confidence_threshold = 0.8
        df = df[df['Center_p'] > confidence_threshold]

        trajectory = get_trajectory(df, n_steps=2000)

        t_shelter = frac_time_spent_in_shelter(trajectory)

        t_investigating = frac_time_spent_investigating(trajectory)
        
        transition_dict = calculate_zone_transitions(trajectory)
        n_sh_co = transition_dict['Shelter to Corridor']
        n_co_sh = transition_dict["Corridor to Shelter"]
        n_co_ch = transition_dict["Corridor to Chamber"]
        n_ch_co = transition_dict["Chamber to Corridor"]

        heatmap_entropy = calculate_heatmap_entropy(trajectory)

        laziness = calculate_laziness(trajectory)
        
        results_list.append({
                'mouse_number': mouse_num,
                'phase': phase,
                't_shelter': t_shelter,
                't_investigating': t_investigating,
                'n_sh_co': n_sh_co,
                'n_co_sh': n_co_sh,
                'n_co_ch': n_co_ch,
                'n_ch_co': n_ch_co,
                'heatmap_entropy': heatmap_entropy,
                'laziness': laziness
            })
        

results_df = pd.DataFrame(results_list)

results_df.to_csv('mouse_analysis_true.csv', index=False)

print("Processing complete. Data saved to mouse_analysis_results.csv")

Processing complete. Data saved to mouse_analysis_results.csv


In [5]:
results_df.head()

,mouse_number,phase,t_shelter,t_investigating,n_sh_co,n_co_sh,n_co_ch,n_ch_co,heatmap_entropy,laziness
0,13,def1,0.5015,0.1260,9,9,8,8,4.157402,0.8345
1,13,def3,0.2165,0.2755,14,14,13,14,4.679773,0.7895
2,14,def1,0.4895,0.1040,13,13,9,9,3.746475,0.8495
3,14,def3,0.3215,0.2595,11,11,12,12,4.166432,0.8260
4,15,def1,0.6880,0.0825,10,10,4,4,2.652491,0.9215
